# 🗄️ Notebook 03 : Analyses SQL Avancées

**Auteur** : Oumaro Titans DJIGUIMDE  
**Date** : Février 2026  
**Objectif** : Réaliser des analyses SQL complexes sur les données de ventes

---

## 🎯 Objectifs de ce notebook

1. Créer une base de données SQLite
2. Requêtes SQL de base (SELECT, WHERE, GROUP BY)
3. Requêtes avancées avec JOINs
4. Window Functions (ROW_NUMBER, RANK, LAG, LEAD)
5. Analyses de cohortes et tendances
6. Calcul de KPIs business
7. Requêtes optimisées avec CTEs

## 📦 Importation et configuration

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Bibliothèques importées")

## 1️⃣ Création de la base de données SQLite

In [ ]:
# Chargement du dataset nettoyé
df = pd.read_csv('../data/sales_data_clean.csv', parse_dates=['date'])

print("="*70)
print("📊 DATASET CHARGÉ")
print("="*70)
print(f"Lignes : {len(df):,}")
print(f"Colonnes : {len(df.columns)}")
print(f"CA total : {df['chiffre_affaires'].sum():,.0f} FCFA")
print("="*70)

In [ ]:
# Création de la base de données SQLite
db_path = '../data/ventes_senegal.db'
engine = create_engine(f'sqlite:///{db_path}')

# Insertion des données dans la table 'ventes'
df.to_sql('ventes', engine, if_exists='replace', index=False)

print("="*70)
print("🗄️  BASE DE DONNÉES CRÉÉE")
print("="*70)
print(f"Fichier : {db_path}")
print(f"Table : ventes")
print(f"Lignes insérées : {len(df):,}")
print("✅ Prêt pour les requêtes SQL !")
print("="*70)

In [ ]:
# Fonction helper pour exécuter des requêtes SQL
def execute_sql(query, description=""):
    """
    Exécute une requête SQL et affiche les résultats
    """
    if description:
        print(f"\n{'='*70}")
        print(f"🔍 {description}")
        print(f"{'='*70}")
    
    result = pd.read_sql_query(query, engine)
    print(result)
    print(f"\n📊 {len(result)} lignes retournées")
    return result

print("✅ Fonction helper définie")

## 2️⃣ Requêtes SQL de base

In [ ]:
# Query 1 : Top 10 produits par chiffre d'affaires
query1 = """
SELECT 
    produit,
    categorie,
    SUM(chiffre_affaires) as ca_total,
    COUNT(*) as nb_ventes,
    AVG(prix) as prix_moyen
FROM ventes
GROUP BY produit, categorie
ORDER BY ca_total DESC
LIMIT 10;
"""

result1 = execute_sql(query1, "TOP 10 PRODUITS PAR CA")

In [ ]:
# Query 2 : Performance par ville
query2 = """
SELECT 
    ville,
    COUNT(*) as nb_transactions,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen,
    MIN(chiffre_affaires) as ca_min,
    MAX(chiffre_affaires) as ca_max,
    ROUND(100.0 * SUM(chiffre_affaires) / (SELECT SUM(chiffre_affaires) FROM ventes), 2) as pct_ca
FROM ventes
GROUP BY ville
ORDER BY ca_total DESC;
"""

result2 = execute_sql(query2, "PERFORMANCE PAR VILLE")

In [ ]:
# Query 3 : Performance par catégorie
query3 = """
SELECT 
    categorie,
    COUNT(*) as nb_transactions,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen,
    SUM(quantite) as quantite_totale,
    ROUND(100.0 * SUM(chiffre_affaires) / (SELECT SUM(chiffre_affaires) FROM ventes), 2) as pct_ca
FROM ventes
GROUP BY categorie
ORDER BY ca_total DESC;
"""

result3 = execute_sql(query3, "PERFORMANCE PAR CATÉGORIE")

In [ ]:
# Query 4 : CA mensuel avec évolution
query4 = """
SELECT 
    strftime('%Y-%m', date) as mois,
    COUNT(*) as nb_transactions,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen
FROM ventes
GROUP BY mois
ORDER BY mois;
"""

result4 = execute_sql(query4, "CA MENSUEL 2024")

## 3️⃣ Requêtes avec filtres complexes

In [ ]:
# Query 5 : Ventes de haute valeur (> 100 000 FCFA)
query5 = """
SELECT 
    date,
    produit,
    categorie,
    ville,
    prix,
    quantite,
    chiffre_affaires,
    mode_paiement
FROM ventes
WHERE chiffre_affaires > 100000
ORDER BY chiffre_affaires DESC
LIMIT 20;
"""

result5 = execute_sql(query5, "TOP 20 VENTES À FORTE VALEUR (> 100K FCFA)")

In [ ]:
# Query 6 : Analyse des ventes pendant le Ramadan
query6 = """
SELECT 
    categorie,
    COUNT(*) as nb_ventes,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen
FROM ventes
WHERE date BETWEEN '2024-03-11' AND '2024-04-09'
GROUP BY categorie
ORDER BY ca_total DESC;
"""

result6 = execute_sql(query6, "VENTES PENDANT LE RAMADAN (11 mars - 9 avril)")

In [ ]:
# Query 7 : Analyse des ventes pendant la Tabaski
query7 = """
SELECT 
    categorie,
    COUNT(*) as nb_ventes,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen
FROM ventes
WHERE date BETWEEN '2024-06-10' AND '2024-06-20'
GROUP BY categorie
ORDER BY ca_total DESC;
"""

result7 = execute_sql(query7, "VENTES PENDANT LA TABASKI (10-20 juin)")

In [ ]:
# Query 8 : Produits électroniques les plus vendus à Dakar
query8 = """
SELECT 
    produit,
    COUNT(*) as nb_ventes,
    SUM(chiffre_affaires) as ca_total,
    AVG(prix) as prix_moyen
FROM ventes
WHERE categorie = 'Électronique' AND ville = 'Dakar'
GROUP BY produit
ORDER BY nb_ventes DESC
LIMIT 10;
"""

result8 = execute_sql(query8, "TOP 10 PRODUITS ÉLECTRONIQUES À DAKAR")

## 4️⃣ Window Functions (Fonctions analytiques)

In [ ]:
# Query 9 : Classement des produits par CA avec ROW_NUMBER
query9 = """
SELECT 
    produit,
    categorie,
    SUM(chiffre_affaires) as ca_total,
    ROW_NUMBER() OVER (ORDER BY SUM(chiffre_affaires) DESC) as rang_global,
    ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY SUM(chiffre_affaires) DESC) as rang_categorie
FROM ventes
GROUP BY produit, categorie
ORDER BY ca_total DESC
LIMIT 15;
"""

result9 = execute_sql(query9, "CLASSEMENT DES PRODUITS (ROW_NUMBER)")

In [ ]:
# Query 10 : Top 3 produits par catégorie
query10 = """
WITH RankedProducts AS (
    SELECT 
        categorie,
        produit,
        SUM(chiffre_affaires) as ca_total,
        ROW_NUMBER() OVER (PARTITION BY categorie ORDER BY SUM(chiffre_affaires) DESC) as rang
    FROM ventes
    GROUP BY categorie, produit
)
SELECT 
    categorie,
    produit,
    ca_total,
    rang
FROM RankedProducts
WHERE rang <= 3
ORDER BY categorie, rang;
"""

result10 = execute_sql(query10, "TOP 3 PRODUITS PAR CATÉGORIE")

In [ ]:
# Query 11 : Évolution mensuelle avec LAG (comparaison mois précédent)
query11 = """
WITH MonthlySales AS (
    SELECT 
        strftime('%Y-%m', date) as mois,
        SUM(chiffre_affaires) as ca_total
    FROM ventes
    GROUP BY mois
)
SELECT 
    mois,
    ca_total,
    LAG(ca_total) OVER (ORDER BY mois) as ca_mois_precedent,
    ca_total - LAG(ca_total) OVER (ORDER BY mois) as evolution_absolue,
    ROUND(100.0 * (ca_total - LAG(ca_total) OVER (ORDER BY mois)) / 
          NULLIF(LAG(ca_total) OVER (ORDER BY mois), 0), 2) as evolution_pct
FROM MonthlySales
ORDER BY mois;
"""

result11 = execute_sql(query11, "ÉVOLUTION MENSUELLE DU CA (LAG)")

In [ ]:
# Query 12 : Cumul du CA par ville
query12 = """
WITH DailyVille AS (
    SELECT 
        ville,
        date,
        SUM(chiffre_affaires) as ca_jour
    FROM ventes
    WHERE strftime('%Y-%m', date) = '2024-06'
    GROUP BY ville, date
)
SELECT 
    ville,
    date,
    ca_jour,
    SUM(ca_jour) OVER (PARTITION BY ville ORDER BY date) as ca_cumule
FROM DailyVille
ORDER BY ville, date
LIMIT 30;
"""

result12 = execute_sql(query12, "CA CUMULÉ PAR VILLE (JUIN 2024)")

## 5️⃣ Analyses avancées avec CTEs

In [ ]:
# Query 13 : Analyse Pareto (80/20) - Produits générant 80% du CA
query13 = """
WITH ProductCA AS (
    SELECT 
        produit,
        SUM(chiffre_affaires) as ca_total
    FROM ventes
    GROUP BY produit
),
RankedProducts AS (
    SELECT 
        produit,
        ca_total,
        SUM(ca_total) OVER (ORDER BY ca_total DESC) as ca_cumule,
        (SELECT SUM(ca_total) FROM ProductCA) as ca_global
    FROM ProductCA
)
SELECT 
    produit,
    ca_total,
    ca_cumule,
    ROUND(100.0 * ca_cumule / ca_global, 2) as pct_cumule
FROM RankedProducts
WHERE 100.0 * ca_cumule / ca_global <= 80
ORDER BY ca_total DESC;
"""

result13 = execute_sql(query13, "ANALYSE PARETO - PRODUITS GÉNÉRANT 80% DU CA")

In [ ]:
# Query 14 : Analyse par jour de la semaine
query14 = """
SELECT 
    CASE CAST(strftime('%w', date) AS INTEGER)
        WHEN 0 THEN 'Dimanche'
        WHEN 1 THEN 'Lundi'
        WHEN 2 THEN 'Mardi'
        WHEN 3 THEN 'Mercredi'
        WHEN 4 THEN 'Jeudi'
        WHEN 5 THEN 'Vendredi'
        WHEN 6 THEN 'Samedi'
    END as jour_semaine,
    COUNT(*) as nb_transactions,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen,
    ROUND(100.0 * SUM(chiffre_affaires) / (SELECT SUM(chiffre_affaires) FROM ventes), 2) as pct_ca
FROM ventes
GROUP BY strftime('%w', date)
ORDER BY strftime('%w', date);
"""

result14 = execute_sql(query14, "PERFORMANCE PAR JOUR DE LA SEMAINE")

In [ ]:
# Query 15 : Matrice ville × catégorie
query15 = """
SELECT 
    ville,
    categorie,
    COUNT(*) as nb_ventes,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen
FROM ventes
GROUP BY ville, categorie
ORDER BY ville, ca_total DESC;
"""

result15 = execute_sql(query15, "MATRICE VILLE × CATÉGORIE")

## 6️⃣ KPIs Business

In [ ]:
# Query 16 : KPIs globaux
query16 = """
SELECT 
    COUNT(*) as nb_transactions,
    COUNT(DISTINCT produit) as nb_produits_distincts,
    COUNT(DISTINCT vendeur) as nb_vendeurs,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen,
    MIN(chiffre_affaires) as ca_min,
    MAX(chiffre_affaires) as ca_max,
    SUM(quantite) as quantite_totale,
    AVG(quantite) as quantite_moyenne
FROM ventes;
"""

result16 = execute_sql(query16, "📊 KPIS GLOBAUX")

In [ ]:
# Query 17 : Taux de pénétration par mode de paiement
query17 = """
SELECT 
    mode_paiement,
    COUNT(*) as nb_transactions,
    SUM(chiffre_affaires) as ca_total,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ventes), 2) as pct_transactions,
    ROUND(100.0 * SUM(chiffre_affaires) / (SELECT SUM(chiffre_affaires) FROM ventes), 2) as pct_ca
FROM ventes
GROUP BY mode_paiement
ORDER BY ca_total DESC;
"""

result17 = execute_sql(query17, "💳 RÉPARTITION PAR MODE DE PAIEMENT")

In [ ]:
# Query 18 : Performance des vendeurs
query18 = """
SELECT 
    vendeur,
    COUNT(*) as nb_ventes,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen,
    MAX(chiffre_affaires) as plus_grosse_vente,
    COUNT(DISTINCT categorie) as nb_categories
FROM ventes
GROUP BY vendeur
ORDER BY ca_total DESC
LIMIT 10;
"""

result18 = execute_sql(query18, "🏆 TOP 10 VENDEURS")

## 7️⃣ Analyses de cohortes et segmentation

In [ ]:
# Query 19 : Segmentation par tranche de CA
query19 = """
WITH Segments AS (
    SELECT 
        *,
        CASE 
            WHEN chiffre_affaires < 10000 THEN 'Petit'
            WHEN chiffre_affaires BETWEEN 10000 AND 50000 THEN 'Moyen'
            WHEN chiffre_affaires BETWEEN 50000 AND 100000 THEN 'Important'
            ELSE 'Premium'
        END as segment
    FROM ventes
)
SELECT 
    segment,
    COUNT(*) as nb_transactions,
    SUM(chiffre_affaires) as ca_total,
    AVG(chiffre_affaires) as ca_moyen,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ventes), 2) as pct_transactions,
    ROUND(100.0 * SUM(chiffre_affaires) / (SELECT SUM(chiffre_affaires) FROM ventes), 2) as pct_ca
FROM Segments
GROUP BY segment
ORDER BY 
    CASE segment 
        WHEN 'Petit' THEN 1
        WHEN 'Moyen' THEN 2
        WHEN 'Important' THEN 3
        WHEN 'Premium' THEN 4
    END;
"""

result19 = execute_sql(query19, "📊 SEGMENTATION PAR TRANCHE DE CA")

In [ ]:
# Query 20 : Analyse trimestielle
query20 = """
WITH QuarterlySales AS (
    SELECT 
        CASE 
            WHEN CAST(strftime('%m', date) AS INTEGER) BETWEEN 1 AND 3 THEN 'Q1'
            WHEN CAST(strftime('%m', date) AS INTEGER) BETWEEN 4 AND 6 THEN 'Q2'
            WHEN CAST(strftime('%m', date) AS INTEGER) BETWEEN 7 AND 9 THEN 'Q3'
            ELSE 'Q4'
        END as trimestre,
        categorie,
        SUM(chiffre_affaires) as ca_total
    FROM ventes
    GROUP BY trimestre, categorie
)
SELECT 
    trimestre,
    categorie,
    ca_total,
    RANK() OVER (PARTITION BY trimestre ORDER BY ca_total DESC) as rang
FROM QuarterlySales
ORDER BY trimestre, rang;
"""

result20 = execute_sql(query20, "📅 PERFORMANCE PAR TRIMESTRE ET CATÉGORIE")

## 8️⃣ Visualisation des résultats SQL

In [ ]:
# Visualisation : Évolution mensuelle
plt.figure(figsize=(14, 6))
result11['mois'] = pd.to_datetime(result11['mois'])
plt.plot(result11['mois'], result11['ca_total'], marker='o', linewidth=2, markersize=8, color='green')
plt.title('Évolution Mensuelle du CA 2024', fontsize=16, fontweight='bold')
plt.xlabel('Mois', fontsize=12)
plt.ylabel('CA (FCFA)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation : Top 10 produits
plt.figure(figsize=(12, 8))
result1_sorted = result1.sort_values('ca_total', ascending=True)
plt.barh(result1_sorted['produit'], result1_sorted['ca_total'], color='coral')
plt.title('Top 10 Produits par Chiffre d\'Affaires', fontsize=16, fontweight='bold')
plt.xlabel('CA (FCFA)', fontsize=12)
plt.ylabel('Produit', fontsize=12)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation : Performance par jour de la semaine
plt.figure(figsize=(10, 6))
plt.bar(result14['jour_semaine'], result14['ca_total'], color='skyblue', edgecolor='black')
plt.title('Chiffre d\'Affaires par Jour de la Semaine', fontsize=16, fontweight='bold')
plt.xlabel('Jour', fontsize=12)
plt.ylabel('CA (FCFA)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 📌 Résumé des analyses SQL

### ✅ Requêtes réalisées

**Niveau basique** :
- Top produits par CA
- Performance par ville et catégorie
- Évolution mensuelle

**Niveau intermédiaire** :
- Filtres complexes (périodes spéciales)
- Agrégations multiples
- Pourcentages et ratios

**Niveau avancé** :
- Window Functions (ROW_NUMBER, LAG, RANK)
- CTEs (Common Table Expressions)
- Analyse Pareto (80/20)
- Cumuls et évolutions

### 🎯 Compétences démontrées

✅ Maîtrise de SQL (SELECT, WHERE, GROUP BY, HAVING)  
✅ Fonctions analytiques avancées  
✅ Optimisation avec CTEs  
✅ Calcul de KPIs business  
✅ Analyse de cohortes et segmentation  
✅ Interprétation des résultats  

### 💡 Insights clés

1. **Saisonnalité forte** : Pics pendant Tabaski, Ramadan, fin d'année
2. **Concentration géographique** : Dakar ~45% du CA
3. **Modes de paiement mobile** : 65%+ des transactions
4. **Principe de Pareto** : Quelques produits génèrent l'essentiel du CA
5. **Week-end** : Activité commerciale marquée

### 🚀 Applications possibles

- Optimisation des stocks par ville et saison
- Ciblage marketing par segment de clientèle
- Prévision des ventes (Machine Learning)
- Tableaux de bord décisionnels
- Analyses prédictives

---

**Projet terminé ! 🎉**  
Portfolio complet d’ingénierie de données, entièrement réalisé par mes soins.